# Exercices 11

> 🧭 **Séance hors-programme.** Cette séance ne fait pas partie de la matière évaluée. Elle reprend tout ce qui a été vu aux séances 7 à 10 et constitue un bon projet personnel pour qui souhaite aller plus loin.

[Télécharger l'exercice](../11_exercice.zip)

# Exercice : équation de la glace en 2D

![](https://images.contenthub.dev/xshiytapvw9a/e404ac0c4deb0f9ef3d3bd6859603bb2/Gornergletscher%20Wanderung%20Rotenboden.jpg?fm=avif&fit=fill&q=45&h=640&w=1140)

*Le Glacier du Gorner vers Zermatt*

## Déformation de la glace

L'objectif de l'exercice est d'implémenter l'équation de la glace en 2D pour simuler l'évolution du glacier du Gorner situé vers Zermatt. En deux dimensions, l'évolution de l'épaisseur de la glace se traduit par

$$\frac{\partial h}{\partial t} = - \frac{\partial q_x}{\partial x}
- \frac{\partial q_y}{\partial y} + b(s), \qquad (1) $$

$$ q_x = - D(h) \frac{\partial s}{\partial x}. \qquad q_y = - D(h) \frac{\partial s}{\partial y}$$

où $h$ est l'épaisseur de glace, $l$ l'altitude du lit rocheux, $s=l+h$ l'altitude de la surface de la glace, $t$ le temps, $b$ le bilan de masse (accumulation moins ablation de glace en surface), et $D$ la diffusivité dynamique de la glace. La non-linéarité vient de la constante de diffusivité $D$, qui dépend elle-même de l'épaisseur locale de la glace $h$ et de la pente de sa surface $||\nabla s||^2 =  (\frac{\partial s}{\partial x})^2 + (\frac{\partial s}{\partial y})^2$ :

$$D(h) = f_\mathrm{d} (\rho g)^3 h^5 ||\nabla s||^2  \qquad (2)$$

où $f_\mathrm{d}$ est une constante physique, $\rho$ la masse volumique de la glace, $g$ l'accélération de la pesanteur.

## Lecture des données (fichiers `topg.dat`, `thk.dat` et `icemask.dat`)

Pour la topographie du lit rocheux, utilisez le fichier `topg.dat` ainsi que le code qui lit le tableau des altitudes (avec une résolution de 100 m). Dans cette première étape, nous utilisons le paramètre `sous_ech` pour travailler avec une résolution plus grossière, ce qui est toujours utile lors de la construction du code. En définissant `sous_ech=2`, nous obtiendrons une résolution de 200 m. Cela réduira la taille de la grille par un facteur de 4, ce qui rendra le code plus rapide. Une fois le code terminé, nous pourrons tester une résolution plus élevée en réglant `sous_ech=1`. Nous lirons également les fichiers `thk.dat`, qui fournissent l'épaisseur contemporaine du glacier (que nous prendrons comme condition initiale en l'absence de données passées), ainsi que `icemask.dat`, qui permet de définir un masque utile pour le calcul du bilan de masse. Voilà le code qui permet de lire les données :

```python
sous_ech = 2
l = np.loadtxt('topg.dat')[::sous_ech,::sous_ech]
h = np.loadtxt('thk.dat')[::sous_ech,::sous_ech]
masque = np.loadtxt('icemask.dat')[::sous_ech,::sous_ech]
```

A partir des données et de la résolution de 100 m, on peut en déduire les paramètres numériques:

```python
ny,nx = l.shape
dx=sous_ech*100
dy=sous_ech*100
Lx = dx*nx
Ly = dy*ny
```

## Forçage climatique

Comme dans le cas 1D, le bilan de masse $b(s)$ est défini en fonction de l'altitude et dépend de trois facteurs : 1) une pente qui contrôle le gradient du bilan de masse $b_\mathrm{grad}$, 2) une valeur maximale pour l'accumulation $b_\mathrm{max}$, et 3) l'altitude de la ligne d'équilibre $s_\mathrm{ELA}$ au-delà de laquelle la glace s'accumule et en deçà de laquelle la glace fond. Mathématiquement, $b(s)$ peut ainsi être écrit comme :

$$b(s) = \min ( b_\mathrm{grad} (s-s_\mathrm{ELA}), b_\mathrm{max} ).$$

Dans cet exercice, la valeur de la ligne d'équilibre $s_\mathrm{ELA}$ varie au cours du temps (afin de faire évoluer le glacier, voir la table des paramètres plus bas). Ensuite, nous implémenterons la formule du bilan de masse ainsi :

```python
    b = np.minimum(b_grad * (s - s_ELA), b_max)
    b = np.where( (b < 0) | (masque > 0.5), b, -10)
```

La seconde ligne ci-dessus utilise un "masque", qui est lu dans les données. Ce masque permet de confiner le glacier du Gorner : en dehors du masque (`masque < 0.5`), le `np.where` remplace le bilan par une ablation forte (-10 m/an) **uniquement là où celui-ci serait positif ou nul**, c'est-à-dire là où de la glace s'accumulerait. Les zones qui sont déjà en ablation gardent leur bilan. Cela empêche la formation d'autres glaciers (non souhaités) dans le domaine modélisé.

## Figure

Pour obtenir une figure ressemblant à celle ci-dessous, nous pouvons utiliser le code suivant, qui permet un affichage simultané de la surface et de l'épaisseur de glace à l'initialisation :

```python
fig, (ax1,ax2) = plt.subplots(1,2,figsize=(8, 6),dpi=200)

f1 = ax1.imshow(s, extent=[0, Lx/1000, 0, Ly/1000],  cmap='terrain', origin='lower', vmin=0, vmax=4800)
cbar = fig.colorbar(f1, ax=ax1, orientation='horizontal', label='Altitude, m')

f2 = ax2.imshow(np.where(h>0,h,np.nan), extent=[0, Lx/1000, 0, Ly/1000], cmap='jet', origin='lower', vmin=0, vmax=1000)
cbar2 = fig.colorbar(f2, ax=ax2, orientation='horizontal', label='Épaisseur de glace, m')
```

et dans la boucle:

```python
clear_output(wait=True)
ax1.cla()
ax2.cla()

ax1.imshow(s, extent=[0, Lx/1000, 0, Ly/1000],  cmap='terrain',origin='lower', vmin=0, vmax=4800)
ax1.set_title('Surface de la glace en ' + str(int(temps)))
ax1.set_xlabel('Distance, km')
ax1.set_ylabel('Distance, km')

ax2.imshow(np.where(h>0,h,np.nan), extent=[0, Lx/1000, 0, Ly/1000], cmap='jet',origin='lower', vmin=0, vmax=1000)
ax2.set_title('Volume de glace : ' + str(round(volume_glace, 2)) + ' $km^3$')

ax2.set_xlabel('Distance, km')
ax2.set_ylabel('Distance, km')

display(fig)
```

Optionnellement, nous avons ici affiché le volume total de glace, qui peut être calculé à partir de l'épaisseur de glace et de la résolution de la grille $dx$.

![](./fig/gorner-mod.png)

*Modèle de l'écoulement de glace du glacier du Gorner.*

## But: Modélisation du retrait récent, et modélisation pronostique jusqu'à 2100

En utilisant les données topographiques du glacier (lit rocheux et épaisseur contemporaine), nous allons modéliser le glacier de 1950 jusqu'à 2100. L'objectif est de reconstruire son retrait passé récent, ainsi qu'un pronostic de son évolution jusqu'en 2100 (avec une forte augmentation des températures matérialisée par une augmentation significative de la ligne d'équilibre). Pour cela, on suppose une ligne d'équilibre de 3200 m fixe jusqu'en 2000, qui augmentera linéairement de 5 m par an (soit 100 m tous les 20 ans) entre 2000 et 2100.

Observez l'évolution du glacier du Gorner en 2020 et comparez votre résultat aux images sur [map.geo.admin.ch](https://map.geo.admin.ch).


| **Paramètre**                                    | **Valeur**            | **Unité** |
|--------------------------------------------------|-----------------------|-----------|
| accélération de la pesanteur, $g$                | 9.81                  | m/s²      |
| masse volumique de la glace, $\rho_\mathrm{ice}$ | 910                   | kg/m³     |
| constante physique, $f_\mathrm{d}$               | $0.25 \times 10^{-16}$ | Pa⁻³ an⁻¹ |
| bilan de masse maximum, $b_\mathrm{max}$         | 50                    | cm/an     |
| gradient du bilan de masse, $b_\mathrm{grad}$    | 0.005                 | an⁻¹      |
| ligne d'équilibre jusqu'en 2000, $s_\mathrm{ELA}$ | 3200                 | m         |
| montée de la ligne d'équilibre après 2000        | 5                     | m/an      |
| pas de temps maximal, $dt_\mathrm{max}$          | 1                     | an        |
| Temps initial                                    | 1950                  | an        |
| Temps final                                      | 2100                  | an        |

> ⚠️ **Attention aux unités !** Les paramètres ne vous sont pas tous donnés dans des unités compatibles entre elles. Convertissez-les explicitement dans votre code, avant la boucle temporelle.



### ✅ **À vous de faire !**